In [0]:
from pyspark.sql.functions import col, lower, initcap, month, year, when, count, regexp_replace
from pyspark.sql.types import DoubleType, IntegerType

In [0]:
# Acessando a tabela delta bronze
df_bronze = spark.table("projeto_profissional_bigdata.datatran_2018.bronze_datatran2018")

In [0]:
df_bronze.printSchema()


In [0]:
#selecionando colunas
df_silver = df_bronze.select(
    "id", 
    "data_inversa",
    "dia_semana",
    "horario",
    "uf",
    "br",
    "km",
    "municipio",
    "causa_acidente",
    "tipo_acidente")

df_silver.show(5)
df_silver.printSchema()

In [0]:
# removendo linhas com valores nulos
df_silver = df_silver.dropna(subset=["uf", "data_inversa", "uf", "br", "km", "municipio", "causa_acidente", "tipo_acidente"])
df_silver.show(5)

In [0]:
# renomeando coluna data_inversa para data
df_silver = df_silver.withColumnRenamed("data_inversa", "data_acidente")

In [0]:
# padronizar nome de colunas
df_silver = df_silver.withColumn("municipio", initcap(col("municipio")))
df_silver = df_silver.withColumn("causa_acidente", lower(col("causa_acidente")))
df_silver = df_silver.withColumn("tipo_acidente", lower(col("tipo_acidente")))
df_silver.show(5)

In [0]:
df_silver = df_silver.withColumn(
    "mes_acidente",
    month(col("data_acidente")))
df_silver.show(5)

In [0]:
# criando coluna ano_acidente
df_silver = df_silver.withColumn(
    "ano_acidente",
    year(col("data_acidente")))

df_silver.show(5)

In [0]:
df_silver = df_silver.withColumn(
    "nome_mes",
    when(col("mes_acidente") == 1, "janeiro")
    .when(col("mes_acidente") == 2, "fevereiro")
    .when(col("mes_acidente") == 3, "março")
    .when(col("mes_acidente") == 4, "abril")
    .when(col("mes_acidente") == 5, "maio")
    .when(col("mes_acidente") == 6, "junho")
    .when(col("mes_acidente") == 7, "julho")
    .when(col("mes_acidente") == 8, "agosto")
    .when(col("mes_acidente") == 9, "setembro")
    .when(col("mes_acidente") == 10, "outubro")
    .when(col("mes_acidente") == 11, "novembro")
    .when(col("mes_acidente") == 12, "dezembro")

)
df_silver.show(5)

In [0]:
df_silver.filter(col("br")=="NA").count()

In [0]:
df_silver = df_silver.withColumn(
    "br",
    when(col("br") == "NA", None)
    .otherwise(col("br"))
)

In [0]:
#Limpando os dados nulos da coluna br
df_silver = df_silver.dropna(subset=["br"])

In [0]:
df_silver = df_silver.withColumn(
    "br",
    col("br").cast(IntegerType())
)

df_silver.printSchema()
df_silver.show(5)

In [0]:
df_silver = df_silver.withColumn(
    "km",
    regexp_replace(col("km"), ",", ".").cast(DoubleType())
)

df_silver.printSchema()
df_silver.show(5)

In [0]:
df_silver.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("projeto_profissional_bigdata.datatran_2018.silver_datatran2018")

In [0]:
df = spark.table("projeto_profissional_bigdata.datatran_2018.silver_datatran2018")
df.show()

In [0]:
%sql
SELECT * FROM projeto_profissional_bigdata.datatran_2018.silver_datatran2018